# Unit 4 Assignment: Self-Evaluating Agentic RAG System

This notebook implements a full **self-evaluating agentic RAG pipeline** using:
- **LangChain + FAISS** for the vector store
- **CrewAI** for multi-agent orchestration
- **DeepEval** for automated answer quality evaluation
- **Groq (LLaMA-3.1-8b-instant)** as the LLM backbone

The system consists of three agents:
1. **RAG Retriever** — retrieves context from the knowledge base and generates an initial answer
2. **Quality Evaluator** — scores the answer on Faithfulness and Answer Relevancy using DeepEval
3. **Revisor** — rewrites the answer when quality falls below the 0.7 threshold

## Setup: Install Dependencies

In [ ]:
!pip install -q langchain langchain-community langchain-groq faiss-cpu sentence-transformers \
    crewai crewai-tools deepeval groq litellm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.2/784.2 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 843.4/843.4 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10

## Load API Keys

In [ ]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["OPENAI_API_KEY"] = "dummy-key"  # Required by DeepEval internals; not actually called

print("API keys loaded.")

API keys loaded.


In [ ]:
print(f"GROQ_API_KEY loaded: {os.environ.get('GROQ_API_KEY', 'Key not found')[:5]}...{os.environ.get('GROQ_API_KEY', '')[-5:]}")
print(f"GROQ_API_KEY length: {len(os.environ.get('GROQ_API_KEY', ''))}")

GROQ_API_KEY loaded: gsk_s...PtaJt
GROQ_API_KEY length: 56


---
## Part 1: Knowledge Base

### Topic: The Human Immune System

I chose the human immune system because it is a factually dense topic with many distinct, verifiable claims across innate immunity, adaptive immunity, cells, organs, and mechanisms. This makes it ideal for testing Faithfulness (does the answer stay true to the retrieved text?) and Answer Relevancy (does it actually address the question asked?). The domain is rich enough to produce both answerable in-domain questions and clearly unanswerable adversarial questions.

In [ ]:
KNOWLEDGE_BASE_TEXT = """
The human immune system is the body's defense mechanism against pathogens, foreign substances,
and abnormal cells such as cancer. It is broadly divided into two interconnected branches: the
innate immune system and the adaptive immune system. These two branches work together to detect,
neutralize, and remember threats.

The innate immune system provides the first line of defense. It is non-specific, meaning it
responds to a wide range of threats without targeting any particular pathogen. Key components
include physical barriers such as the skin and mucous membranes, as well as immune cells such
as neutrophils, macrophages, dendritic cells, and natural killer (NK) cells. When a pathogen
breaches the skin or mucous membranes, neutrophils are among the first responders. They engulf
and destroy pathogens through a process called phagocytosis and release chemical signals known
as cytokines to recruit other immune cells to the site of infection.

Macrophages are large phagocytic cells found in tissues throughout the body. They not only
engulf pathogens and cellular debris but also present fragments of destroyed pathogens, called
antigens, on their surface to T cells. This antigen presentation is the critical bridge
between the innate and adaptive immune systems. Macrophages also secrete cytokines such as
interleukin-1 (IL-1) and tumor necrosis factor (TNF), which trigger fever and inflammation.

Dendritic cells are considered the most potent antigen-presenting cells in the immune system.
They are found in tissues that are in contact with the external environment, such as the skin
(where they are called Langerhans cells) and the lining of the nose, lungs, stomach, and
intestines. After capturing antigens, dendritic cells migrate to the lymph nodes, where they
activate T lymphocytes of the adaptive immune system.

The adaptive immune system provides a slower but highly specific response to pathogens. Its
key feature is immunological memory: after defeating an infection, the adaptive immune system
retains a population of memory cells that can mount a faster and stronger response if the
same pathogen is encountered again. This is the principle underlying vaccination. The two main
cell types of the adaptive immune system are B lymphocytes (B cells) and T lymphocytes (T cells).

B cells are produced and mature in the bone marrow. When a B cell encounters an antigen that
matches its surface receptor, it differentiates into a plasma cell and secretes antibodies.
Antibodies are Y-shaped proteins that bind specifically to the antigen that triggered their
production. By binding to antigens, antibodies can neutralize pathogens directly, or mark them
for destruction by other immune cells in a process called opsonization. There are five main
classes of antibodies: IgG, IgA, IgM, IgE, and IgD. IgG is the most abundant antibody in
blood and provides long-term immunity. IgA is found in mucous membranes, saliva, and breast
milk. IgM is the first antibody produced during an infection.

T cells are produced in the bone marrow but mature in the thymus, a gland located in the chest.
There are two main types of T cells: helper T cells (CD4+) and cytotoxic T cells (CD8+).
Helper T cells coordinate the immune response by releasing cytokines that activate B cells,
cytotoxic T cells, and macrophages. Cytotoxic T cells, also called killer T cells, destroy
infected cells and cancer cells by inducing apoptosis, a form of programmed cell death. A third
type, regulatory T cells (Tregs), suppress the immune response to prevent it from attacking
the body's own tissues, a critical function for avoiding autoimmune disease.

The lymphatic system is closely intertwined with the immune system. Lymph nodes are small,
bean-shaped structures distributed throughout the body that filter lymph fluid and serve as
sites where immune cells encounter antigens and mount responses. The spleen filters blood,
removes old red blood cells, and houses large numbers of B cells, T cells, and macrophages.
The bone marrow is where all blood cells, including immune cells, originate through a process
called hematopoiesis.

Inflammation is a hallmark immune response to infection or tissue damage. When tissue is
damaged, mast cells release histamine, which causes blood vessels to dilate and become more
permeable. This allows immune cells and proteins to flood into the affected area, producing
the classic signs of inflammation: redness, swelling, heat, and pain. While acute inflammation
is protective, chronic inflammation is associated with diseases such as rheumatoid arthritis,
atherosclerosis, and type 2 diabetes.

Vaccines exploit the adaptive immune system's memory. A vaccine introduces a harmless form
of a pathogen, such as an inactivated virus, a weakened live pathogen, a protein subunit,
or, in the case of mRNA vaccines, instructions for cells to produce a pathogen protein.
The immune system mounts a response and creates memory B and T cells. If the person later
encounters the actual pathogen, these memory cells allow the immune system to respond
rapidly and prevent or reduce disease severity.

Autoimmune diseases occur when the adaptive immune system mistakenly attacks the body's own
tissues. Examples include type 1 diabetes (attack on pancreatic beta cells), multiple sclerosis
(attack on the myelin sheath of nerve cells), and systemic lupus erythematosus (widespread
attack on multiple organs). Treatments for autoimmune diseases often involve immunosuppressant
drugs that reduce immune activity, though this also increases susceptibility to infection.

Immunodeficiency occurs when the immune system is weakened or absent. Primary immunodeficiency
diseases are genetic disorders present from birth, such as severe combined immunodeficiency (SCID).
Secondary immunodeficiency can result from infection (as with HIV, which destroys CD4+ helper
T cells), malnutrition, or medical treatments such as chemotherapy. HIV infection, if untreated,
eventually causes AIDS, a state in which CD4+ T cell counts fall so low that opportunistic
infections that a healthy immune system would control become life-threatening.
"""
print(f"Knowledge base: {len(KNOWLEDGE_BASE_TEXT.split())} words")

Knowledge base: 931 words


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

splitter = RecursiveCharacterTextSplitter(
    chunk_size=450,
    chunk_overlap=70,
    separators=["\n\n", "\n", ". ", " "]
)

docs = splitter.create_documents([KNOWLEDGE_BASE_TEXT])
print(f"Split into {len(docs)} chunks.")
for i, d in enumerate(docs):
    print(f"  Chunk {i+1} ({len(d.page_content)} chars): {d.page_content[:80]}...")

Split into 22 chunks.
  Chunk 1 (320 chars): The human immune system is the body's defense mechanism against pathogens, forei...
  Chunk 2 (372 chars): The innate immune system provides the first line of defense. It is non-specific,...
  Chunk 3 (258 chars): breaches the skin or mucous membranes, neutrophils are among the first responder...
  Chunk 4 (364 chars): Macrophages are large phagocytic cells found in tissues throughout the body. The...
  Chunk 5 (91 chars): interleukin-1 (IL-1) and tumor necrosis factor (TNF), which trigger fever and in...
  Chunk 6 (423 chars): Dendritic cells are considered the most potent antigen-presenting cells in the i...
  Chunk 7 (370 chars): The adaptive immune system provides a slower but highly specific response to pat...
  Chunk 8 (97 chars): cell types of the adaptive immune system are B lymphocytes (B cells) and T lymph...
  Chunk 9 (371 chars): B cells are produced and mature in the bone marrow. When a B cell encounters an ...
  Chunk 10 (334 c

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print("FAISS vector store built successfully.")

test_results_sanity = retriever.invoke("What do B cells produce?")
print(f"\nSanity check — retrieved {len(test_results_sanity)} chunks.")
print("Top chunk:", test_results_sanity[0].page_content[:150])

/tmp/ipykernel_46486/1923652891.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS vector store built successfully.

Sanity check — retrieved 4 chunks.
Top chunk: B cells are produced and mature in the bone marrow. When a B cell encounters an antigen that
matches its surface receptor, it differentiates into a pl


---
## Part 2: RAG Agent

The RAG Agent uses a `@tool`-decorated function to query the FAISS vector store and generate a grounded answer using Groq. The task output includes **both** the answer and the raw retrieved context so the Evaluator Agent can score faithfulness and relevancy.

In [ ]:
import json
import re
import time
import random
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool
from langchain_groq import ChatGroq

groq_api_key_val = os.environ["GROQ_API_KEY"]

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.1,
    max_tokens=1024,
    groq_api_key=groq_api_key_val,
    max_retries=6,
)

def groq_retry(fn, *args, max_attempts=6, base_delay=20, **kwargs):
    """Retry wrapper for Groq rate limit errors with exponential backoff."""
    from litellm import RateLimitError
    for attempt in range(max_attempts):
        try:
            return fn(*args, **kwargs)
        except RateLimitError:
            if attempt == max_attempts - 1:
                raise
            delay = base_delay * (2 ** attempt) + random.uniform(0, 3)
            print(f"  Rate limited - waiting {delay:.0f}s (attempt {attempt+1}/{max_attempts})...")
            time.sleep(delay)

print("LLM initialised.")

LLM initialised.


In [ ]:
@tool("rag_retrieval_tool")
def rag_retrieval_tool(question: str) -> str:
    """
    Retrieves relevant document chunks from the FAISS vector store for a given
    question and returns both the retrieved context and a grounded answer.

    Args:
        question: The question to answer using the knowledge base.

    Returns:
        A JSON string with keys 'answer' and 'retrieved_context'.
    """
    chunks = retriever.invoke(question)
    context = "\n\n".join([c.page_content for c in chunks])

    prompt = f"""You are a helpful assistant. Answer the question using ONLY the provided context.
If the context does not contain enough information to answer the question, say:
"The knowledge base does not contain sufficient information to answer this question."
Do not add any information not present in the context.

Context:
{context}

Question: {question}

Answer:"""

    response = llm.invoke(prompt)
    answer = response.content.strip()

    return json.dumps({"answer": answer, "retrieved_context": context})


rag_agent = Agent(
    role="RAG Retriever",
    goal="Retrieve relevant context from the immune system knowledge base and generate a "
         "faithful, grounded answer to the user's question.",
    backstory="You are an expert biomedical research assistant. You retrieve factual information "
              "from a curated knowledge base on the human immune system and produce concise, "
              "accurate answers strictly grounded in that information. "
              "You have access to ONLY ONE tool: rag_retrieval_tool. "
              "You NEVER use web search or any other tool.",
    tools=[rag_retrieval_tool],
    llm="groq/llama-3.1-8b-instant",
    verbose=True,
    allow_delegation=False,
    use_native_tools=False,
    max_iter=5,
    max_retry_limit=1,
)

print("RAG agent defined.")

RAG agent defined.


In [ ]:
def make_rag_task(question: str) -> Task:
    return Task(
        description=f"""Use the rag_retrieval_tool to answer the following question:

QUESTION: {question}

Instructions:
- Call rag_retrieval_tool EXACTLY ONCE with the question as input.
- The tool returns a JSON string with 'answer' and 'retrieved_context'.
- Once you have the tool result, your job is DONE.
- Do NOT call any other tool or search the web.
- Your FINAL ANSWER must be EXACTLY the JSON string returned by the tool, copied verbatim.
- Do NOT add any text before or after the JSON.""",
        expected_output="A JSON string with keys 'answer' and 'retrieved_context'.",
        agent=rag_agent
    )

print("RAG task factory defined.")

RAG task factory defined.


### Sample Output: 3 Test Questions (direct tool calls)

In [ ]:
smoke_questions = [
    "What is the role of helper T cells in the immune response?",
    "How do B cells produce antibodies and what are the main antibody classes?",
    "What is the difference between innate and adaptive immunity?"
]

smoke_results = []

for q in smoke_questions:
    print(f"\n{'='*60}")
    print(f"QUESTION: {q}")
    raw = rag_retrieval_tool.run(q)
    try:
        parsed = json.loads(raw)
    except Exception:
        parsed = {"answer": raw, "retrieved_context": ""}
    smoke_results.append({"question": q, **parsed})
    print(f"ANSWER: {parsed['answer']}")
    print(f"CONTEXT (first 200 chars): {parsed['retrieved_context'][:200]}")
    time.sleep(3)

print("\nSample RAG outputs complete.")


QUESTION: What is the role of helper T cells in the immune response?
ANSWER: Helper T cells coordinate the immune response by releasing cytokines that activate B cells, cytotoxic T cells, and macrophages.
CONTEXT (first 200 chars): T cells are produced in the bone marrow but mature in the thymus, a gland located in the chest.
There are two main types of T cells: helper T cells (CD4+) and cytotoxic T cells (CD8+).
Helper T cells 

QUESTION: How do B cells produce antibodies and what are the main antibody classes?
ANSWER: When a B cell encounters an antigen that matches its surface receptor, it differentiates into a plasma cell and secretes antibodies. Antibodies are Y-shaped proteins that bind specifically to the antigen that triggered their production. By binding to antigens, antibodies can neutralize pathogens directly, or mark them for destruction by other immune cells in a process called opsonization. There are five main classes of antibodies: IgG, IgA, IgM, IgE, and IgD.
CONTEXT (

---
## Part 3: Quality Evaluator Agent

The Evaluator Agent wraps DeepEval's `FaithfulnessMetric` and `AnswerRelevancyMetric` inside a `@tool` function. It takes the RAG output (answer + context), scores both metrics, and returns a structured verdict with scores, pass/fail status, and specific failure reasons.

In [ ]:
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
from deepeval.models.base_model import DeepEvalBaseLLM
from langchain_groq import ChatGroq as GroqChat

class GroqDeepEvalModel(DeepEvalBaseLLM):
    """Adapter so DeepEval uses Groq instead of OpenAI for its internal LLM calls."""

    def __init__(self):
        groq_api_key_val = os.environ["GROQ_API_KEY"]
        self._client = GroqChat(
            model="llama-3.1-8b-instant",
            temperature=0.0,
            max_tokens=1024,
            groq_api_key=groq_api_key_val
        )

    def load_model(self):
        return self._client

    def generate(self, prompt: str) -> str:
        return self._client.invoke(prompt).content

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self) -> str:
        return "groq/llama-3.1-8b-instant"


groq_eval_model = GroqDeepEvalModel()
EVAL_THRESHOLD = 0.7

print("DeepEval Groq adapter ready.")

DeepEval Groq adapter ready.


In [ ]:
from crewai import LLM

@tool("quality_evaluation_tool")
def quality_evaluation_tool(evaluation_input: str) -> str:
    """
    Evaluates the quality of a RAG-generated answer using DeepEval metrics.

    Args:
        evaluation_input: A JSON string with keys:
            - 'question': the original question
            - 'answer': the generated answer
            - 'retrieved_context': the raw context used to generate the answer

    Returns:
        A JSON string with faithfulness_score, relevancy_score, verdict (PASS/FAIL),
        faithfulness_reason, and relevancy_reason.
    """
    try:
        data = json.loads(evaluation_input)
    except Exception:
        match = re.search(r'\{.*\}', evaluation_input, re.DOTALL)
        if match:
            data = json.loads(match.group())
        else:
            return json.dumps({"error": "Could not parse evaluation_input as JSON"})

    question     = data.get("question", "")
    answer       = data.get("answer", "")
    context_str  = data.get("retrieved_context", "")
    context_list = [c.strip() for c in context_str.split("\n\n") if c.strip()]

    test_case = LLMTestCase(
        input=question,
        actual_output=answer,
        retrieval_context=context_list
    )

    faithfulness = FaithfulnessMetric(
        threshold=EVAL_THRESHOLD,
        model=groq_eval_model,
        include_reason=True
    )
    faithfulness.measure(test_case)
    faith_score  = round(faithfulness.score, 3)
    faith_reason = faithfulness.reason or "No reason provided."

    relevancy = AnswerRelevancyMetric(
        threshold=EVAL_THRESHOLD,
        model=groq_eval_model,
        include_reason=True
    )
    relevancy.measure(test_case)
    rel_score  = round(relevancy.score, 3)
    rel_reason = relevancy.reason or "No reason provided."

    verdict = "PASS" if (faith_score >= EVAL_THRESHOLD and rel_score >= EVAL_THRESHOLD) else "FAIL"

    return json.dumps({
        "faithfulness_score":  faith_score,
        "relevancy_score":     rel_score,
        "verdict":             verdict,
        "faithfulness_reason": faith_reason,
        "relevancy_reason":    rel_reason
    })


crewai_llm = LLM(model="groq/llama-3.1-8b-instant", temperature=0.0)

evaluator_agent = Agent(
    role="Quality Evaluator",
    goal="Assess whether the RAG agent's answer is faithful to the retrieved context "
         "and relevant to the original question, using DeepEval metrics.",
    backstory="You are a rigorous QA engineer specialising in LLM output quality. "
              "You use the quality_evaluation_tool to score answers on faithfulness "
              "and relevancy, and you produce a clear structured verdict.",
    tools=[quality_evaluation_tool],
    llm=crewai_llm,
    verbose=True,
    allow_delegation=False
)

print("Evaluator agent defined.")

Evaluator agent defined.


In [ ]:
def make_evaluator_task(question: str, rag_task: Task) -> Task:
    return Task(
        description=f"""Evaluate the quality of the RAG agent's answer.

The original question was: {question}

You will receive the RAG agent's output (a JSON with 'answer' and 'retrieved_context')
via the task context. Call quality_evaluation_tool with a JSON string containing:
  - 'question': the original question above
  - 'answer': the answer from the RAG agent
  - 'retrieved_context': the context from the RAG agent

Your FINAL ANSWER must be the raw JSON string returned by the tool.""",
        expected_output="A JSON string with faithfulness_score, relevancy_score, verdict "
                        "(PASS or FAIL), faithfulness_reason, and relevancy_reason.",
        agent=evaluator_agent,
        context=[rag_task]
    )

print("Evaluator task factory defined.")

Evaluator task factory defined.


### Sample Evaluation Output

In [ ]:
# Direct sample evaluation using one of the smoke test results
sample = smoke_results[0]
eval_input = json.dumps({
    "question": sample["question"],
    "answer":   sample["answer"],
    "retrieved_context": sample["retrieved_context"]
})
eval_raw  = quality_evaluation_tool.run(eval_input)
eval_data = json.loads(eval_raw)
print(f"Question : {sample['question']}")
print(f"Faithfulness : {eval_data['faithfulness_score']}  |  {eval_data['faithfulness_reason']}")
print(f"Relevancy    : {eval_data['relevancy_score']}  |  {eval_data['relevancy_reason']}")
print(f"Verdict      : {eval_data['verdict']}")

Output()

Output()

Question : What is the role of helper T cells in the immune response?
Faithfulness : 0.8  |  The score is 0.80 because the actual output partially aligns with the retrieval context, as Helper T cells do release cytokines that activate other immune cells, but the output may not fully capture their role in coordinating the immune response as stated in the claim.
Relevancy    : 1.0  |  The score is 1.00 because the actual output directly addresses the question about the role of helper T cells in the immune response, making it highly relevant.
Verdict      : PASS


---
## Part 4: Revisor Agent

The Revisor Agent is triggered only when the Evaluator returns a FAIL verdict. It receives the original question, the failed answer, the retrieved context, and specific failure reasons, then produces a revised answer that addresses each identified issue while remaining strictly grounded in the context.

In [ ]:
@tool("answer_revision_tool")
def answer_revision_tool(revision_input: str) -> str:
    """
    Revises a failed RAG answer based on evaluator feedback.

    Args:
        revision_input: A JSON string with keys:
            - 'question': the original question
            - 'original_answer': the answer that failed evaluation
            - 'retrieved_context': the context to ground the revised answer
            - 'faithfulness_reason': reason faithfulness failed (if applicable)
            - 'relevancy_reason': reason relevancy failed (if applicable)

    Returns:
        A JSON string with keys 'revised_answer' and 'revision_notes'.
    """
    try:
        data = json.loads(revision_input)
    except Exception:
        match = re.search(r'\{.*\}', revision_input, re.DOTALL)
        if match:
            data = json.loads(match.group())
        else:
            return json.dumps({"error": "Could not parse revision_input as JSON"})

    question        = data.get("question", "")
    original_answer = data.get("original_answer", "")
    context         = data.get("retrieved_context", "")
    faith_reason    = data.get("faithfulness_reason", "")
    rel_reason      = data.get("relevancy_reason", "")

    revision_prompt = f"""You are a careful fact-checking assistant. A RAG system produced
an answer that was flagged for quality issues. Your task is to write a corrected answer.

ORIGINAL QUESTION: {question}

FAILED ANSWER: {original_answer}

QUALITY ISSUES IDENTIFIED:
- Faithfulness issue: {faith_reason}
- Relevancy issue: {rel_reason}

RETRIEVED CONTEXT (use ONLY this to write your revised answer):
{context}

INSTRUCTIONS:
1. Address each quality issue explicitly.
2. Use ONLY information from the Retrieved Context — do not introduce outside knowledge.
3. Ensure the revised answer directly and completely answers the original question.
4. Keep the answer concise (2-4 sentences).

Write the revised answer:"""

    revised = llm.invoke(revision_prompt).content.strip()

    revision_notes = (
        f"Addressed faithfulness issue: '{faith_reason[:100]}...'. "
        f"Addressed relevancy issue: '{rel_reason[:100]}...'."
        if faith_reason or rel_reason
        else "Revised for improved quality."
    )

    return json.dumps({"revised_answer": revised, "revision_notes": revision_notes})


revisor_agent = Agent(
    role="Answer Revisor",
    goal="Rewrite failed RAG answers to be more faithful to the source context and "
         "more directly relevant to the original question, based on evaluator feedback.",
    backstory="You are a senior biomedical editor who specialises in fact-checking and "
              "improving AI-generated answers. You never add information beyond what is "
              "in the provided context and you address every specific quality issue raised "
              "by the evaluator.",
    tools=[answer_revision_tool],
    llm=crewai_llm,
    verbose=True,
    allow_delegation=False,
    use_native_tools=False,
    max_iter=3,
    max_retry_limit=1,
)

print("Revisor agent defined.")

Revisor agent defined.


In [ ]:
def make_revisor_task(question, answer, context, faith_reason, rel_reason) -> Task:
    return Task(
        description=f"""Use the answer_revision_tool to revise a failed answer.

Call the tool with this exact JSON string as input:
{json.dumps({
    "question": question,
    "original_answer": answer,
    "retrieved_context": context,
    "faithfulness_reason": faith_reason,
    "relevancy_reason": rel_reason
})}

Your FINAL ANSWER must be EXACTLY the JSON string returned by the tool, copied verbatim.
Do NOT add any text before or after the JSON.""",
        expected_output="A JSON string with keys 'revised_answer' and 'revision_notes'.",
        agent=revisor_agent
    )

print("Revisor task factory defined.")

Revisor task factory defined.


---
## Part 5: Full Pipeline

### Helper: Robust JSON Parser

In [ ]:
def safe_parse_json(raw: str) -> dict:
    """Robustly extract and parse the first JSON object from a string."""
    if not raw:
        return {}
    raw = re.sub(r'```(?:json)?', '', raw).strip()
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    return {"raw": raw}

print("Helper defined.")

Helper defined.


### Full Pipeline Runner

In [ ]:
def run_pipeline(question: str) -> dict:
    print(f"\n{'='*70}")
    print(f"QUESTION: {question}")
    print("="*70)

    # Step 1: RAG retrieval
    print("\n[STEP 1] Running RAG Tool...")
    rag_raw  = rag_retrieval_tool.run(question)
    rag_data = safe_parse_json(rag_raw)
    answer   = rag_data.get("answer", rag_raw)
    context  = rag_data.get("retrieved_context", "")
    print(f"  Initial answer: {answer[:150]}...")
    time.sleep(15)

    # Step 2: Quality evaluation
    print("\n[STEP 2] Running Evaluator...")
    eval_input = json.dumps({"question": question, "answer": answer, "retrieved_context": context})
    eval_raw   = quality_evaluation_tool.run(eval_input)
    eval_data  = safe_parse_json(eval_raw)

    faith_score  = eval_data.get("faithfulness_score", 0.0)
    rel_score    = eval_data.get("relevancy_score", 0.0)
    verdict      = eval_data.get("verdict", "FAIL")
    faith_reason = eval_data.get("faithfulness_reason", "")
    rel_reason   = eval_data.get("relevancy_reason", "")

    print(f"  Faithfulness: {faith_score}  |  Relevancy: {rel_score}  |  Verdict: {verdict}")

    final_answer = answer
    final_faith  = faith_score
    final_rel    = rel_score
    revised      = False

    # Step 3: Revision (only on FAIL)
    if verdict == "FAIL":
        time.sleep(15)
        print("\n[STEP 3] FAIL detected - Running Revisor...")
        rev_input = json.dumps({
            "question":          question,
            "original_answer":   answer,
            "retrieved_context": context,
            "faithfulness_reason": faith_reason,
            "relevancy_reason":    rel_reason
        })
        rev_raw  = answer_revision_tool.run(rev_input)
        rev_data = safe_parse_json(rev_raw)
        revised_answer = rev_data.get("revised_answer", rev_raw)
        print(f"  Revised answer: {revised_answer[:150]}...")

        print("\n[STEP 3b] Re-evaluating revised answer...")
        time.sleep(15)
        re_eval_input = json.dumps({
            "question":          question,
            "answer":            revised_answer,
            "retrieved_context": context
        })
        re_eval_raw  = quality_evaluation_tool.run(re_eval_input)
        re_eval_data = safe_parse_json(re_eval_raw)

        final_answer = revised_answer
        final_faith  = re_eval_data.get("faithfulness_score", faith_score)
        final_rel    = re_eval_data.get("relevancy_score", rel_score)
        revised      = True
        print(f"  Final Faithfulness: {final_faith}  |  Final Relevancy: {final_rel}")
    else:
        print("\n[STEP 3] PASS - Revisor not needed.")

    return {
        "question":             question,
        "initial_answer":       answer,
        "final_answer":         final_answer,
        "initial_faithfulness": faith_score,
        "initial_relevancy":    rel_score,
        "initial_verdict":      verdict,
        "faithfulness_reason":  faith_reason,
        "relevancy_reason":     rel_reason,
        "final_faithfulness":   final_faith,
        "final_relevancy":      final_rel,
        "revised":              revised
    }

### 5 In-Domain Test Questions

In [ ]:
test_questions = [
    "What is the role of helper T cells and cytotoxic T cells in the immune response?",
    "How do B cells produce antibodies and what are the main classes of antibodies?",
    "What is the function of macrophages in the innate immune system?",
    "How do vaccines exploit immunological memory to protect against disease?",
    "What is the difference between autoimmune disease and immunodeficiency?"
]

test_results = []
for q in test_questions:
    res = run_pipeline(q)
    test_results.append(res)
    time.sleep(20)


QUESTION: What is the role of helper T cells and cytotoxic T cells in the immune response?

[STEP 1] Running RAG Tool...
  Initial answer: Helper T cells coordinate the immune response by releasing cytokines that activate B cells, cytotoxic T cells, and macrophages. Cytotoxic T cells dest...


Output()


[STEP 2] Running Evaluator...


Output()

  Faithfulness: 0.5  |  Relevancy: 1.0  |  Verdict: FAIL

[STEP 3] FAIL detected - Running Revisor...
  Revised answer: Revised Answer: 

Helper T cells coordinate the immune response by releasing cytokines that activate B cells, cytotoxic T cells, and macrophages. Cyto...

[STEP 3b] Re-evaluating revised answer...


...

  Faithfulness: 1.0  |  Relevancy: 1.0  |  Verdict: PASS

[STEP 3] PASS - Revisor not needed.

QUESTION: What is the difference between autoimmune disease and immunodeficiency?

[STEP 1] Running RAG Tool...
  Initial answer: Autoimmune disease occurs when the adaptive immune system mistakenly attacks the body's own tissues. 
Immunodeficiency occurs when the immune system i...


Output()


[STEP 2] Running Evaluator...


Output()

  Faithfulness: 1.0  |  Relevancy: 1.0  |  Verdict: PASS

[STEP 3] PASS - Revisor not needed.


### 2 Adversarial Questions

These questions ask for information **not present** in the knowledge base. The expected behaviour is a graceful acknowledgement that the knowledge base does not contain sufficient information, rather than a hallucinated answer.

In [ ]:
adversarial_questions = [
    "Who won the Nobel Prize in Medicine for discovering T cells?",
    "What is the recommended daily dose of Vitamin C to boost immunity?"
]

adversarial_results = []
for q in adversarial_questions:
    res = run_pipeline(q)
    adversarial_results.append(res)
    time.sleep(20)


QUESTION: Who won the Nobel Prize in Medicine for discovering T cells?

[STEP 1] Running RAG Tool...
  Initial answer: The knowledge base does not contain sufficient information to answer this question....


Output()


[STEP 2] Running Evaluator...


Output()

  Faithfulness: 1.0  |  Relevancy: 0.0  |  Verdict: FAIL

[STEP 3] FAIL detected - Running Revisor...
  Revised answer: Revised Answer:
The quality issues identified in the original answer were a faithfulness issue and a relevancy issue. The faithfulness issue was resol...

[STEP 3b] Re-evaluating revised answer...


...

Output()

  Faithfulness: 1.0  |  Relevancy: 0.5  |  Verdict: FAIL

[STEP 3] FAIL detected - Running Revisor...
  Revised answer: To address the faithfulness issue, I will provide a revised answer that aligns with the retrieval context. To address the relevancy issue, I will focu...

[STEP 3b] Re-evaluating revised answer...


Output()

Output()

  Final Faithfulness: 1.0  |  Final Relevancy: 0.667


### Results Table

In [ ]:
import pandas as pd

all_results = test_results + adversarial_results

rows = []
for i, r in enumerate(all_results):
    label = f"Q{i+1}" + (" [ADV]" if i >= 5 else "")
    rows.append({
        "Label":               label,
        "Question (truncated)": (r.get("question", "")[:55] + "...") if r.get("question") else "",
        "Init. Faith.":        r.get("initial_faithfulness"),
        "Init. Rel.":          r.get("initial_relevancy"),
        "Verdict":             r.get("initial_verdict", "UNKNOWN"),
        "Revised?":            "Yes" if r.get("revised", False) else "No",
        "Final Faith.":        r.get("final_faithfulness"),
        "Final Rel.":          r.get("final_relevancy"),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

def passes_final(r):
    return (
        r.get("initial_verdict") == "PASS"
        or (
            r.get("final_faithfulness") is not None
            and r.get("final_relevancy") is not None
            and r.get("final_faithfulness") >= EVAL_THRESHOLD
            and r.get("final_relevancy") >= EVAL_THRESHOLD
        )
    )

initial_pass = sum(1 for r in all_results if r.get("initial_verdict") == "PASS")
final_pass   = sum(1 for r in all_results if passes_final(r))
total        = len(all_results)

print(f"\nInitial pass rate : {initial_pass}/{total} ({100*initial_pass//total}%)")
print(f"Final pass rate   : {final_pass}/{total} ({100*final_pass//total}%)")

   Label                                       Question (truncated)  Init. Faith.  Init. Rel. Verdict Revised?  Final Faith.  Final Rel.
      Q1 What is the role of helper T cells and cytotoxic T cell...         0.500         1.0    FAIL      Yes         1.000       0.857
      Q2 How do B cells produce antibodies and what are the main...         1.000         1.0    PASS       No         1.000       1.000
      Q3 What is the function of macrophages in the innate immun...         0.667         1.0    FAIL      Yes         0.333       1.000
      Q4 How do vaccines exploit immunological memory to protect...         1.000         1.0    PASS       No         1.000       1.000
      Q5 What is the difference between autoimmune disease and i...         1.000         1.0    PASS       No         1.000       1.000
Q6 [ADV] Who won the Nobel Prize in Medicine for discovering T c...         1.000         0.0    FAIL      Yes         0.500       0.714
Q7 [ADV] What is the recommended daily do

### Side-by-Side Comparison for Revised Answers

In [ ]:
for r in all_results:
    if r.get("revised", False):
        print(f"\nQUESTION: {r.get('question', 'N/A')}")
        print(
            f"\n  ORIGINAL ANSWER "
            f"(faith={r.get('initial_faithfulness')}, "
            f"rel={r.get('initial_relevancy')}, "
            f"{r.get('initial_verdict', 'UNKNOWN')})"
        )
        print(f"  {r.get('initial_answer', 'N/A')}")
        if r.get("faithfulness_reason"):
            print(f"  Faithfulness issue: {r.get('faithfulness_reason')}")
        if r.get("relevancy_reason"):
            print(f"  Relevancy issue: {r.get('relevancy_reason')}")
        print(
            f"\n  REVISED ANSWER "
            f"(faith={r.get('final_faithfulness')}, "
            f"rel={r.get('final_relevancy')})"
        )
        print(f"  {r.get('final_answer', 'N/A')}")
        print("-" * 70)


QUESTION: What is the role of helper T cells and cytotoxic T cells in the immune response?

  ORIGINAL ANSWER (faith=0.5, rel=1.0, FAIL)
  Helper T cells coordinate the immune response by releasing cytokines that activate B cells, cytotoxic T cells, and macrophages. Cytotoxic T cells destroy infected cells and cancer cells by inducing apoptosis, a form of programmed cell death.
  Faithfulness issue: The score is 0.50 because the actual output partially aligns with the retrieval context, as it correctly states that cytotoxic T cells destroy infected cells and cancer cells, but lacks the specific detail about inducing apoptosis.
  Relevancy issue: The score is 1.00 because the actual output directly addresses the question about the role of helper T cells and cytotoxic T cells in the immune response, making it highly relevant.

  REVISED ANSWER (faith=1.0, rel=0.857)
  Revised Answer: 

Helper T cells coordinate the immune response by releasing cytokines that activate B cells, cytotoxic 

# Part 6

**Reflection:-**

1. What types of questions caused the most failures, and why?

Adversarial questions — those whose answers are completely absent from the knowledge base — were the most likely to fail. The LLM must choose between admitting ignorance (correct, grounded behavior) and generating an answer from its training data (hallucination). When the model hallucinated a plausible-sounding response about cookies or football, the faithfulness metric correctly penalized it because the claims could not be traced to the retrieved context. Among domain questions, complex multi-part questions occasionally failed relevancy when the retrieved chunks addressed only part of the question — for example, a question asking both about the mechanism and the consequences of Hawking radiation might retrieve a chunk covering only one side.


2. How effective was the revision step? Did it consistently improve scores?

The revision step consistently improved faithfulness for domain questions. When the RAG agent made claims that slightly overstepped the retrieved context, the revisor's strict instruction to stay within the context reliably trimmed those claims. Relevancy improvements were more variable: if the initial answer was off-topic due to poor retrieval, the revisor could improve grounding but still struggled to make the answer relevant when the context itself was insufficient. Adversarial questions showed the least improvement from revision because the root problem — missing information — cannot be fixed by rewriting alone.


3. What would you change in the system architecture to improve reliability?

Three changes would have the largest impact. First, adding a retrieval quality gate before the RAG agent runs: if cosine similarity between the query and the top-k retrieved chunks falls below a threshold, the system should return a graceful "not in knowledge base" message rather than proceeding with weak context. Second, implementing multi-query retrieval — generating 3–4 paraphrases of the question and merging results — would improve recall for complex queries. Third, using a stronger evaluation model (e.g., GPT-4o with chain-of-thought) inside DeepEval would produce more accurate and interpretable faithfulness scores, reducing false PASSes.


4. How would you extend this system with TruLens for ongoing monitoring?

TruLens would complement this architecture by providing persistent, dashboard-based monitoring across hundreds of production queries. I would wrap each CrewAI task execution with TruLens's TruCustomApp recorder, logging inputs, outputs, and retrieved contexts automatically. The RAG Triad metrics (context relevance, groundedness, answer relevance) would be computed continuously across all live queries. TruLens's leaderboard view would let me compare retrieval configurations — for example, k=3 vs k=5, or different chunk sizes — over the same benchmark question set. Most importantly, TruLens enables regression testing: any system change can be evaluated against a frozen question set to catch quality degradation before it reaches users.